In [ ]:
import os
DATA_ROOT = os.environ.get('WRH_DATA_ROOT', 'G:/WRH_data')  # local data folder; see README
# pip install gcsfs zarr
import os, numpy as np, pandas as pd, xarray as xr, gcsfs

SIC_OUT = rf'{DATA_ROOT}/new_climate/siconc'
os.makedirs(SIC_OUT, exist_ok=True)
LAT_KEEP = 55.0                      # keep only north of 55°N

P = 'gs://cmip6/CMIP6'
STORES = {
 ('ACCESS','historical'): f'{P}/CMIP/CSIRO-ARCCSS/ACCESS-CM2/historical/r1i1p1f1/SImon/{{V}}/gn/{{H}}/',
 ('ACCESS','ssp126'):     f'{P}/ScenarioMIP/CSIRO-ARCCSS/ACCESS-CM2/ssp126/r1i1p1f1/SImon/{{V}}/gn/{{S}}/',
 ('ACCESS','ssp585'):     f'{P}/ScenarioMIP/CSIRO-ARCCSS/ACCESS-CM2/ssp585/r1i1p1f1/SImon/{{V}}/gn/{{S}}/',
 ('ECEARTH','historical'):f'{P}/CMIP/EC-Earth-Consortium/EC-Earth3/historical/r1i1p1f1/SImon/{{V}}/gn/{{E}}/',
 ('ECEARTH','ssp126'):    f'{P}/ScenarioMIP/EC-Earth-Consortium/EC-Earth3/ssp126/r1i1p1f1/SImon/{{V}}/gn/{{E}}/',
 ('ECEARTH','ssp585'):    f'{P}/ScenarioMIP/EC-Earth-Consortium/EC-Earth3/ssp585/r1i1p1f1/SImon/{{V}}/gn/{{E}}/',
}
VERS = {'H':'v20200817', 'S':'v20210317', 'E':'v20200918'}   # verified siconc versions
fs = gcsfs.GCSFileSystem(token='anon')

def latname(ds):
    for c in ('latitude','lat','nav_lat'):
        if c in ds.coords or c in ds.variables: return c
    raise KeyError(f'latitude coordinate not found: {list(ds.coords)}')

def fetch(model, exp, var):
    fo = os.path.join(SIC_OUT, f'{var}_{model}_{exp}.nc')
    if os.path.exists(fo): print(f'[SKIP] {os.path.basename(fo)}'); return
    path = STORES[(model, exp)].format(V=var, **VERS)
    try:
        ds = xr.open_zarr(fs.get_mapper(path), consolidated=True)
    except Exception as e:                      # version may differ by variable; auto-detect
        base = '/'.join(path.rstrip('/').split('/')[:-1])
        vs = sorted(x.split('/')[-1] for x in fs.ls(base.replace('gs://','')) if '/v20' in x)
        if not vs: print(f'[MISS] {model} {exp} {var}: {type(e).__name__}'); return
        ds = xr.open_zarr(fs.get_mapper(base + '/' + vs[-1] + '/'), consolidated=True)
        print(f'  (falling back to version {vs[-1]})')
    ds = ds.sel(time=slice('1985-01-01','2100-12-31'))
    ln = latname(ds); lat2 = ds[ln]
    jd, idd = lat2.dims                          # (j, i) dim names of 2-D coords
    keep = (lat2.max(dim=idd) >= LAT_KEEP).values
    ds = ds.isel({jd: np.flatnonzero(keep)})
    ds = ds[[var, ln, ('longitude' if 'longitude' in ds.variables else 'lon')]] \
         if var in ds else ds
    ds.load().to_netcdf(fo, encoding={var: {'zlib': True, 'complevel': 4}})
    print(f'[DONE] {model} {exp} {var}  {dict(ds.sizes)}  {os.path.getsize(fo)/1e6:.0f} MB')

for var in ('siconc','sithick'):
    for (m, e) in STORES:
        fetch(m, e, var)


In [ ]:
# ══════ Cell S0 — rebuild Arctic domain definition (independent of Arctic notebook)══════
import os, glob, warnings, numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

BASE    = rf'{DATA_ROOT}/new_climate/wave'
ARC_OUT = rf'{DATA_ROOT}/new_climate/wave/_arctic'
SIC_OUT = rf'{DATA_ROOT}/new_climate/siconc'
os.makedirs(SIC_OUT, exist_ok=True)

MODELS = {'ACCESS':  {'his': rf'{BASE}\ACCESS\CDFAC_his',
                      'ssp126': rf'{BASE}\ACCESS\CDFAC1_126',
                      'ssp585': rf'{BASE}\ACCESS\CDFAC1_585'},
          'ECEARTH': {'his': rf'{BASE}\ec_earth\cdfac1\his',
                      'ssp126': rf'{BASE}\ec_earth\cdfac1\126',
                      'ssp585': rf'{BASE}\ec_earth\cdfac1\585'}}
MDL_LABEL = {'ACCESS':'ACCESS-CM2', 'ECEARTH':'EC-Earth3'}
TAGS = {'his': (1985, 2014), 'ssp126': (2015, 2100), 'ssp585': (2015, 2100)}

LAT0 = 60.0
SECTORS = {'Barents-Kara':      (66.5, 82,   15,  100),
           'Laptev-E.Siberian': (66.5, 82,  100,  180),
           'Chukchi-Beaufort':  (66.5, 82, -180, -120),
           'Canadian Arctic':   (66.5, 82, -120,  -45),
           'Greenland-Fram':    (66.5, 82,  -45,   15),
           'Central Arctic':    (82,   90, -180,  180)}

# ── static masks ──
_f0 = sorted(glob.glob(os.path.join(MODELS['ACCESS']['his'], 'ww3.*_hs.nc')))[0]
_d  = xr.open_dataset(_f0)
LAT, LON = _d.latitude.values, _d.longitude.values
MAP = _d['MAPSTA'].values; _d.close()

lon180 = ((LON + 180) % 360) - 180
LA, LO  = np.meshgrid(LAT, lon180, indexing='ij')
OCEAN2D = np.isin(MAP, [1, 7])
INDOM   = OCEAN2D & (LA >= LAT0)

sec = np.full(LA.shape, '', dtype=object)
for name, (S, N, W, E) in SECTORS.items():
    sec[(LA >= S) & (LA < N) & (LO >= W) & (LO < E) & OCEAN2D] = name

FLAT  = np.flatnonzero(INDOM.ravel())
NP_   = len(FLAT)
P_LAT = LA.ravel()[FLAT]
P_LON = LO.ravel()[FLAT]                      # ← longitudes needed by S2
P_SEC = sec.ravel()[FLAT]
P_W   = np.cos(np.deg2rad(P_LAT)).astype(np.float32)
print(f'ocean cells in domain {NP_}  (0.5°, ≥{LAT0:.0f}°N)')

# ── load Cell A2 cache from disk (no rerun)──
ARC = {}
for m in MODELS:
    for t in TAGS:
        f = os.path.join(ARC_OUT, f'arctic_{m}_{t}.npz')
        if os.path.exists(f):
            ARC[(m, t)] = np.load(f); print(f'  [OK]   arctic_{m}_{t}.npz')
        else:
            print(f'  [MISSING] arctic_{m}_{t}.npz  ← Cell A2 not finished; S3/S4 unavailable')


In [ ]:
from scipy.spatial import cKDTree

print('Downloaded files:')
for f in sorted(glob.glob(os.path.join(SIC_OUT, '*.nc'))):
    print(f'  {os.path.basename(f):34s} {os.path.getsize(f)/1e6:7.0f} MB')

def latname(ds):
    for c in ('latitude','lat','nav_lat'):
        if c in ds.coords or c in ds.variables: return c
    raise KeyError(f'latitude coordinate not found: {list(ds.coords)}')

def lonname(ds):
    for c in ('longitude','lon','nav_lon'):
        if c in ds.coords or c in ds.variables: return c
    raise KeyError(f'longitude coordinate not found: {list(ds.coords)}')

def xyz(lat, lon):
    la, lo = np.deg2rad(lat), np.deg2rad(lon)
    return np.stack([np.cos(la)*np.cos(lo), np.cos(la)*np.sin(lo), np.sin(la)], -1)

TGT = xyz(P_LAT, P_LON)                      # target: 27375 Arctic wave grid cells

def to_wave_grid(model, exp, var, maxdist_km=150):
    fp = os.path.join(SIC_OUT, f'{var}_{model}_{exp}.nc')
    ds = xr.open_dataset(fp)
    slat = ds[latname(ds)].values.ravel()
    slon = ((ds[lonname(ds)].values.ravel() + 180) % 360) - 180
    v    = ds[var].values.reshape(ds.sizes['time'], -1).astype(np.float32)
    good = np.isfinite(slat) & np.isfinite(slon) & np.isfinite(v).any(0)
    dist, nn = cKDTree(xyz(slat[good], slon[good])).query(TGT, k=1)
    far = dist * 6371.0 > maxdist_km                       # chord × Earth radius ≈ distance (km)
    out = v[:, good][:, nn]
    out[:, far] = np.nan
    t = pd.to_datetime(ds['time'].values)
    if var == 'siconc' and str(ds[var].attrs.get('units','%')).strip() in ('%','percent'):
        out = out / 100.0                                  # normalise to 0–1
    ds.close()
    print(f'  {model:8s} {exp:11s} {var:8s} {out.shape}  dropped beyond max dist {int(far.sum()):5d} pts  '
          f'range [{np.nanmin(out):.3f}, {np.nanmax(out):.3f}]')
    return pd.DataFrame({'year': t.year, 'month': t.month}), out

SIC = {}
print('\nRemapping:')
for m in ('ACCESS','ECEARTH'):
    for e in ('historical','ssp126','ssp585'):
        for var in ('siconc','sithick'):
            try:    SIC[(m, e, var)] = to_wave_grid(m, e, var)
            except Exception as ex: print(f'  [SKIP] {m} {e} {var}: {type(ex).__name__} {ex}')
print(f'\nSIC ready: {len(SIC)} sets')


In [ ]:
TAGMAP = {'his':'historical', 'ssp126':'ssp126', 'ssp585':'ssp585'}

def crossing(xb, yb, level=0.5):
    for i in range(len(xb)-1):
        if np.isfinite(yb[i]) and np.isfinite(yb[i+1]) and (yb[i]-level)*(yb[i+1]-level) <= 0:
            t = (level-yb[i]) / (yb[i+1]-yb[i]+1e-12)
            return xb[i] + t*(xb[i+1]-xb[i])
    return np.nan

def pair_up(model, var='siconc', tags=None):
    if tags is None: tags = [t for t in TAGS if (model, t) in ARC]
    rec = []
    for tg in tags:
        key = (model, TAGMAP[tg], var)
        if (model, tg) not in ARC or key not in SIC: continue
        z = ARC[(model, tg)]
        f = z['nvalid'].astype(np.float32) / z['ndays'][:, None]
        wy, wm = z['ym']//100, z['ym']%100
        idx, vals = SIC[key]
        lut = {(int(r.year), int(r.month)): i for i, r in idx.iterrows()}
        for k in range(len(wy)):
            j = lut.get((int(wy[k]), int(wm[k])))
            if j is not None: rec.append(np.stack([vals[j], f[k]]))
    if not rec: raise RuntimeError('no months to pair')
    A = np.concatenate([r[None] for r in rec])
    x, y = A[:,0].ravel(), A[:,1].ravel()
    ok = np.isfinite(x) & np.isfinite(y)
    print(f'  {model:8s} {var:8s} paired {len(rec):4d} months, valid samples {ok.sum():,}')
    return x[ok], y[ok]

RES = {}
for m in ('ACCESS','ECEARTH'):
    have = [t for t in TAGS if (m, t) in ARC]
    if not have: print(f'{m}: no ARC cache, skipping'); continue
    print(f'{m}: using tags = {have}')
    for var, hi in (('siconc', 1.0), ('sithick', 3.0)):
        try: x, y = pair_up(m, var)
        except Exception as e: print(f'  {m} {var} skipped: {e}'); continue
        edges = np.linspace(0, hi, 51); cen = (edges[:-1]+edges[1:])/2
        k  = np.digitize(x, edges) - 1
        ok = (k >= 0) & (k < len(cen))
        prof = np.array([y[ok][k[ok]==i].mean() if (k[ok]==i).sum() > 50 else np.nan
                         for i in range(len(cen))])
        cnt  = np.array([int((k[ok]==i).sum()) for i in range(len(cen))])
        c50  = crossing(cen, prof)
        RES[(m, var)] = (cen, prof, cnt, c50)
        lo  = cen[np.nanargmax(prof < 0.9)] if np.any(prof < 0.9) else np.nan
        hi_ = cen[np.nanargmax(prof < 0.1)] if np.any(prof < 0.1) else np.nan
        print(f'  → f=0.5 crossing at {c50:.3f}   transition band [{lo:.3f}, {hi_:.3f}]  width {hi_-lo:.3f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=110)
for ax, var, xlab, hi in [(axes[0],'siconc','Sea-ice concentration (fraction)',1.0),
                          (axes[1],'sithick','Sea-ice thickness (m)',3.0)]:
    drew = False
    for m, c in zip(('ACCESS','ECEARTH'), ('#2b7bba','#c8452b')):
        if (m,var) not in RES: continue
        cen, prof, cnt, c50 = RES[(m,var)]
        ax.plot(cen, prof, color=c, lw=2.2, label=f'{MDL_LABEL[m]}  (50% @ {c50:.2f})')
        ax.axvline(c50, color=c, ls=':', lw=1.2); drew = True
    if var == 'siconc':
        for ref, lab in [(0.15,'15%'), (0.25,'25%'), (0.50,'50%')]:
            ax.axvline(ref, color='#9aa0a6', lw=.8, ls='--')
            ax.text(ref, 1.02, lab, ha='center', fontsize=8, color='#6b7178')
    ax.axhline(.5, color='#4a4a4a', lw=.8)
    ax.set(xlabel=xlab, ylabel='Fraction of month with active wave computation',
           xlim=(0, hi), ylim=(-.03, 1.05))
    ax.grid(alpha=.25, lw=.5)
    if drew: ax.legend(fontsize=9, loc='upper right')
fig.suptitle('Inferring the WW3 ice cut-off: wave-model activity vs. sea-ice state', fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
def refine(model, var='siconc', hi=0.12, nb=48, stable=True):
    x, y = pair_up(model, var)
    edges = np.linspace(0, hi, nb+1); cen = (edges[:-1]+edges[1:])/2
    k = np.digitize(x, edges) - 1; ok = (k>=0)&(k<nb)
    prof = np.array([y[ok][k[ok]==i].mean() if (k[ok]==i).sum()>30 else np.nan for i in range(nb)])
    cnt  = np.array([int((k[ok]==i).sum()) for i in range(nb)])
    for c, p, n in zip(cen, prof, cnt):
        if n > 30: print(f'  siconc {c:.4f}  f={p:.3f}  n={n:,}')
    return cen, prof

print('ACCESS low-end refined bins:'); refine('ACCESS')


In [ ]:
xs, ys = pair_up('ACCESS','siconc'); xt, _ = pair_up('ACCESS','sithick')
n = min(len(xs), len(xt))
eb_c = np.linspace(0,0.3,16); eb_t = np.linspace(0,1.5,16)
kc = np.digitize(xs[:n], eb_c)-1; kt = np.digitize(xt[:n], eb_t)-1
M = np.full((15,15), np.nan)
for i in range(15):
    for j in range(15):
        s = (kc==i)&(kt==j)
        if s.sum() > 30: M[j,i] = ys[:n][s].mean()
fig, ax = plt.subplots(figsize=(6.4,5.4))
im = ax.imshow(M, origin='lower', cmap='viridis', vmin=0, vmax=1, aspect='auto',
               extent=[0,0.3,0,1.5])
ax.set(xlabel='Sea-ice concentration', ylabel='Sea-ice thickness (m)',
       title='Wave-active fraction — which axis controls?')
plt.colorbar(im, label='f'); plt.tight_layout(); plt.show()


In [ ]:
x, y = pair_up('ACCESS','siconc')
edges = np.linspace(0, 0.05, 51); cen = (edges[:-1]+edges[1:])/2
k = np.digitize(x, edges)-1; ok = (k>=0)&(k<50)
for i in range(50):
    n = int((k[ok]==i).sum())
    if n > 30:
        print(f'  siconc {cen[i]:.4f}   f = {y[ok][k[ok]==i].mean():.3f}   n = {n:,}')
